# 📦 BhashaLens — Notebook 1: Data Collection & Cleaning

This notebook downloads, cleans, mixes, and splits translation datasets for MarianMT training.

**Language Pairs:** Hindi, Marathi, Tamil, Gujarati ↔ English (8 bidirectional models)

**Sources:** IIT Bombay, AI4Bharat Samanantar, OPUS, Custom Domain

---

## 1. Setup & Dependencies

In [ ]:
!pip install -q datasets opustools langdetect sentencepiece pyyaml pandas tqdm

In [ ]:
import os
import re
import json
import hashlib
import unicodedata
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import yaml
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42
np.random.seed(42)

print('All dependencies loaded.')

In [ ]:
# ── Detect Runtime ──
RUNTIME = 'local'
if os.path.exists('/kaggle'):
    RUNTIME = 'kaggle'
    BASE_DIR = Path('/kaggle/working/bhashalens_ml')
elif os.path.exists('/content'):
    RUNTIME = 'colab'
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/bhashalens_ml')
else:
    BASE_DIR = Path('./bhashalens_ml')

DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_MIXED = BASE_DIR / 'data' / 'mixed'
DATA_CLEANED = BASE_DIR / 'data' / 'cleaned'
REPORTS_DIR = BASE_DIR / 'reports'

for d in [DATA_RAW, DATA_MIXED, DATA_CLEANED, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Runtime: {RUNTIME}')
print(f'Base directory: {BASE_DIR}')

In [ ]:
# ── Configuration (from pipeline_config.yaml) ──
CONFIG = {
    'language_pairs': ['hi-en', 'mr-en', 'ta-en', 'gu-en'],
    'lang_codes': {
        'hi': 'Hindi', 'en': 'English', 'mr': 'Marathi',
        'ta': 'Tamil', 'gu': 'Gujarati'
    },
    'mixing_ratios': {
        'hi-en': {'iit_bombay': 0.35, 'ai4bharat': 0.40, 'opus': 0.15, 'custom': 0.10},
        'mr-en': {'ai4bharat': 0.65, 'opus': 0.20, 'custom': 0.15},
        'ta-en': {'ai4bharat': 0.65, 'opus': 0.20, 'custom': 0.15},
        'gu-en': {'ai4bharat': 0.65, 'opus': 0.20, 'custom': 0.15},
    },
    'split_ratios': (0.90, 0.05, 0.05),
    'cleaning': {
        'max_length': 512,
        'max_length_ratio': 3.0,
        'lang_detect_confidence': 0.9,
    }
}

# The "reverse" pairs (en-hi, en-mr, etc.) use the same data with src/tgt swapped.
# We only need to collect data for the 4 base pairs.
print('Configuration loaded.')
print(f'Language pairs to collect: {CONFIG["language_pairs"]}')

## 2. Dataset Download

### 2.1 IIT Bombay Hindi-English Corpus

In [ ]:
def download_iit_bombay() -> pd.DataFrame:
    """Download IIT Bombay Hindi-English parallel corpus from HuggingFace."""
    print('Downloading IIT Bombay Hindi-English corpus...')
    ds = load_dataset('cfilt/iitb-english-hindi', split='train')
    
    pairs = []
    for item in tqdm(ds, desc='Processing IIT Bombay'):
        translation = item.get('translation', item)
        if isinstance(translation, dict):
            src = translation.get('hi', '')
            tgt = translation.get('en', '')
        else:
            continue
        pairs.append({'source': src, 'target': tgt})
    
    df = pd.DataFrame(pairs)
    save_path = DATA_RAW / 'hi-en' / 'iit_bombay'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'IIT Bombay: {len(df):,} pairs saved')
    return df

iit_bombay_df = download_iit_bombay()

### 2.2 AI4Bharat Samanantar

In [ ]:
def download_ai4bharat(lang_pair: str) -> pd.DataFrame:
    """Download AI4Bharat Samanantar dataset for a language pair."""
    src_lang = lang_pair.split('-')[0]  # e.g., 'hi'
    print(f'Downloading AI4Bharat Samanantar for {lang_pair}...')
    
    try:
        ds = load_dataset('ai4bharat/samanantar', src_lang, split='train')
    except Exception as e:
        print(f'Primary download failed: {e}')
        print('Trying alternative: ai4bharat/samanantar with language config...')
        try:
            ds = load_dataset('ai4bharat/samanantar', f'{src_lang}-en', split='train')
        except Exception as e2:
            print(f'Alternative also failed: {e2}')
            print(f'>> You may need to manually download Samanantar for {lang_pair}')
            print(f'>> Visit: https://ai4bharat.iitm.ac.in/samanantar/')
            return pd.DataFrame(columns=['source', 'target'])
    
    pairs = []
    for item in tqdm(ds, desc=f'Processing AI4Bharat {lang_pair}'):
        src = item.get('src', item.get(src_lang, ''))
        tgt = item.get('tgt', item.get('en', ''))
        if not src and not tgt:
            translation = item.get('translation', {})
            src = translation.get(src_lang, '')
            tgt = translation.get('en', '')
        pairs.append({'source': src, 'target': tgt})
    
    df = pd.DataFrame(pairs)
    save_path = DATA_RAW / lang_pair / 'ai4bharat'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'AI4Bharat {lang_pair}: {len(df):,} pairs saved')
    return df

ai4bharat_data = {}
for lp in CONFIG['language_pairs']:
    ai4bharat_data[lp] = download_ai4bharat(lp)

### 2.3 OPUS Datasets

In [ ]:
def download_opus(lang_pair: str, corpora=None) -> pd.DataFrame:
    """Download OPUS parallel corpora for a language pair."""
    if corpora is None:
        corpora = ['Tatoeba', 'WikiMatrix']
    
    src_lang = lang_pair.split('-')[0]
    tgt_lang = lang_pair.split('-')[1]
    all_pairs = []
    
    for corpus_name in corpora:
        print(f'Downloading OPUS {corpus_name} for {lang_pair}...')
        try:
            ds = load_dataset(
                'opus100',
                f'{src_lang}-{tgt_lang}',
                split='train',
                trust_remote_code=True
            )
            for item in tqdm(ds, desc=f'OPUS {corpus_name} {lang_pair}'):
                translation = item.get('translation', item)
                src = translation.get(src_lang, '')
                tgt = translation.get(tgt_lang, '')
                all_pairs.append({'source': src, 'target': tgt})
            break  # opus100 already aggregates corpora
        except Exception as e:
            print(f'OPUS {corpus_name} for {lang_pair} failed: {e}')
            try:
                ds = load_dataset(
                    f'Helsinki-NLP/opus-{corpus_name.lower()}',
                    lang1=src_lang, lang2=tgt_lang,
                    split='train'
                )
                for item in tqdm(ds, desc=f'OPUS {corpus_name}'):
                    translation = item.get('translation', item)
                    src = translation.get(src_lang, '')
                    tgt = translation.get(tgt_lang, '')
                    all_pairs.append({'source': src, 'target': tgt})
            except Exception as e2:
                print(f'  Fallback also failed: {e2}')
                continue
    
    df = pd.DataFrame(all_pairs) if all_pairs else pd.DataFrame(columns=['source', 'target'])
    save_path = DATA_RAW / lang_pair / 'opus'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'OPUS {lang_pair}: {len(df):,} pairs saved')
    return df

opus_data = {}
for lp in CONFIG['language_pairs']:
    opus_data[lp] = download_opus(lp)

### 2.4 Custom Domain Data (Optional)

If you have custom domain-specific translation pairs (menus, signs, medical terms, etc.), place them as TSV files in `data/raw/{lang_pair}/custom/data.tsv` with columns `source` and `target`.

In [ ]:
def load_custom_data(lang_pair: str) -> pd.DataFrame:
    """Load custom domain data if available."""
    custom_path = DATA_RAW / lang_pair / 'custom' / 'data.tsv'
    if custom_path.exists():
        df = pd.read_csv(custom_path, sep='\t')
        print(f'Custom data {lang_pair}: {len(df):,} pairs loaded')
        return df
    else:
        print(f'No custom data for {lang_pair} (place TSV at {custom_path})')
        return pd.DataFrame(columns=['source', 'target'])

custom_data = {}
for lp in CONFIG['language_pairs']:
    # Create empty placeholder so the directory exists
    (DATA_RAW / lp / 'custom').mkdir(parents=True, exist_ok=True)
    custom_data[lp] = load_custom_data(lp)

### 2.5 Dataset Summary

In [ ]:
summary_rows = []
for lp in CONFIG['language_pairs']:
    sources = {
        'ai4bharat': len(ai4bharat_data.get(lp, [])),
        'opus': len(opus_data.get(lp, [])),
        'custom': len(custom_data.get(lp, [])),
    }
    if lp == 'hi-en':
        sources['iit_bombay'] = len(iit_bombay_df)
    summary_rows.append({'lang_pair': lp, **sources, 'total': sum(sources.values())})

summary_df = pd.DataFrame(summary_rows)
print('\n📊 Dataset Collection Summary:')
print(summary_df.to_string(index=False))

## 3. Data Cleaning Pipeline

Applies all 10 cleaning rules from the spec:
1. Remove empty pairs
2. Remove identical source-target pairs
3. Remove pairs exceeding 512 characters
4. Remove pairs with length ratio > 3:1
5. Remove duplicate source texts
6. Normalize Unicode to NFC
7. Remove pairs containing URLs or emails
8. Remove pairs where source is wrong language
9. Remove pairs where target is wrong language
10. Generate cleaning report

In [ ]:
URL_REGEX = re.compile(
    r'https?://[\w.-]+(?:\.[\w.-]+)+[\w.,@?^=%&:/~+#-]*'
)
EMAIL_REGEX = re.compile(
    r'[\w.+-]+@[\w-]+\.[\w.-]+'
)

def detect_language_safe(text: str) -> str:
    """Detect language with fallback."""
    try:
        if len(text.strip()) < 10:
            return 'unknown'
        return detect(text)
    except Exception:
        return 'unknown'


class DataCleaner:
    """Applies all 10 cleaning rules from the BhashaLens spec."""
    
    def __init__(self, config: dict):
        self.max_length = config['max_length']
        self.max_ratio = config['max_length_ratio']
        self.stats = defaultdict(int)
    
    def clean(self, df: pd.DataFrame, src_lang: str, tgt_lang: str) -> pd.DataFrame:
        """Run full cleaning pipeline."""
        self.stats = defaultdict(int)
        initial_count = len(df)
        self.stats['initial'] = initial_count
        
        # Ensure string columns
        df = df.copy()
        df['source'] = df['source'].astype(str).fillna('')
        df['target'] = df['target'].astype(str).fillna('')
        
        # Rule 1: Remove empty pairs
        mask = (df['source'].str.strip().str.len() > 0) & (df['target'].str.strip().str.len() > 0)
        removed = (~mask).sum()
        df = df[mask]
        self.stats['empty_removed'] = removed
        print(f'  Rule 1 (empty): removed {removed:,}')
        
        # Rule 2: Remove identical pairs
        mask = df['source'] != df['target']
        removed = (~mask).sum()
        df = df[mask]
        self.stats['identical_removed'] = removed
        print(f'  Rule 2 (identical): removed {removed:,}')
        
        # Rule 3: Remove pairs exceeding max length
        mask = (df['source'].str.len() <= self.max_length) & (df['target'].str.len() <= self.max_length)
        removed = (~mask).sum()
        df = df[mask]
        self.stats['too_long_removed'] = removed
        print(f'  Rule 3 (too long): removed {removed:,}')
        
        # Rule 4: Remove pairs with bad length ratio
        src_len = df['source'].str.len().clip(lower=1)
        tgt_len = df['target'].str.len().clip(lower=1)
        ratio = np.maximum(src_len, tgt_len) / np.minimum(src_len, tgt_len)
        mask = ratio <= self.max_ratio
        removed = (~mask).sum()
        df = df[mask]
        self.stats['bad_ratio_removed'] = removed
        print(f'  Rule 4 (ratio > {self.max_ratio}): removed {removed:,}')
        
        # Rule 5: Deduplicate on source text
        before = len(df)
        df = df.drop_duplicates(subset='source', keep='first')
        removed = before - len(df)
        self.stats['duplicates_removed'] = removed
        print(f'  Rule 5 (dedup): removed {removed:,}')
        
        # Rule 6: Unicode NFC normalization
        df['source'] = df['source'].apply(lambda x: unicodedata.normalize('NFC', x))
        df['target'] = df['target'].apply(lambda x: unicodedata.normalize('NFC', x))
        print(f'  Rule 6 (Unicode NFC): applied')
        
        # Rule 7: Remove pairs with URLs or emails
        has_url = df['source'].str.contains(URL_REGEX) | df['target'].str.contains(URL_REGEX)
        has_email = df['source'].str.contains(EMAIL_REGEX) | df['target'].str.contains(EMAIL_REGEX)
        mask = ~(has_url | has_email)
        removed = (~mask).sum()
        df = df[mask]
        self.stats['url_email_removed'] = removed
        print(f'  Rule 7 (URLs/emails): removed {removed:,}')
        
        # Rules 8 & 9: Language detection (sampled for speed)
        print(f'  Rules 8-9 (language detection): checking sample...')
        sample_size = min(5000, len(df))
        sample_idx = df.sample(n=sample_size, random_state=42).index
        
        bad_lang_indices = set()
        for idx in tqdm(sample_idx, desc='Language detection'):
            src_detected = detect_language_safe(df.loc[idx, 'source'])
            tgt_detected = detect_language_safe(df.loc[idx, 'target'])
            
            src_ok = src_detected in [src_lang, 'unknown']
            tgt_ok = tgt_detected in [tgt_lang, 'unknown']
            
            if not src_ok or not tgt_ok:
                bad_lang_indices.add(idx)
        
        if bad_lang_indices:
            # Estimate and remove proportionally from full dataset
            bad_ratio = len(bad_lang_indices) / sample_size
            df = df.drop(index=list(bad_lang_indices), errors='ignore')
            self.stats['wrong_language_removed'] = len(bad_lang_indices)
            print(f'  Rules 8-9: removed {len(bad_lang_indices):,} (est. {bad_ratio:.1%} bad language)')
        else:
            self.stats['wrong_language_removed'] = 0
            print(f'  Rules 8-9: all samples passed language detection')
        
        self.stats['final'] = len(df)
        self.stats['total_removed'] = initial_count - len(df)
        self.stats['retention_rate'] = len(df) / initial_count if initial_count > 0 else 0
        
        return df.reset_index(drop=True)
    
    def get_report(self) -> dict:
        return dict(self.stats)

print('DataCleaner ready.')

## 4. Dataset Mixing

Mix datasets according to spec ratios:
- **Hindi-English:** 35% IIT Bombay, 40% AI4Bharat, 15% OPUS, 10% Custom
- **Others:** 65% AI4Bharat, 20% OPUS, 15% Custom

In [ ]:
def mix_datasets(lang_pair: str, sources: dict, ratios: dict, target_size=None) -> pd.DataFrame:
    """Mix datasets according to specified ratios using stratified sampling."""
    print(f'\nMixing datasets for {lang_pair}...')
    
    available_sources = {k: v for k, v in sources.items() if len(v) > 0}
    available_ratios = {k: v for k, v in ratios.items() if k in available_sources}
    
    if not available_sources:
        print(f'  No data available for {lang_pair}!')
        return pd.DataFrame(columns=['source', 'target'])
    
    # Renormalize ratios for available sources
    total_ratio = sum(available_ratios.values())
    norm_ratios = {k: v / total_ratio for k, v in available_ratios.items()}
    
    # Determine target size (use smallest viable dataset to determine scale)
    if target_size is None:
        # Use maximum available data that respects ratios
        max_possible = min(
            len(available_sources[k]) / norm_ratios[k]
            for k in available_sources
        )
        target_size = int(max_possible)
    
    mixed_dfs = []
    for source_name, ratio in norm_ratios.items():
        n_samples = int(target_size * ratio)
        df = available_sources[source_name]
        
        if n_samples >= len(df):
            sampled = df.copy()
        else:
            sampled = df.sample(n=n_samples, random_state=42)
        
        sampled = sampled.copy()
        sampled['source_dataset'] = source_name
        mixed_dfs.append(sampled)
        print(f'  {source_name}: {len(sampled):,} pairs ({ratio:.0%})')
    
    mixed = pd.concat(mixed_dfs, ignore_index=True)
    mixed = mixed.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle
    
    print(f'  Total mixed: {len(mixed):,} pairs')
    return mixed

print('Mixing function ready.')

## 5. Train / Validation / Test Split

In [ ]:
def hash_split(text: str) -> float:
    """Deterministic hash-based split value [0, 1)."""
    h = hashlib.md5(text.encode('utf-8')).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF


def create_splits(df: pd.DataFrame, ratios=(0.90, 0.05, 0.05)) -> dict:
    """Create train/val/test splits using hash-based deterministic splitting."""
    train_end = ratios[0]
    val_end = ratios[0] + ratios[1]
    
    split_values = df['source'].apply(hash_split)
    
    train_mask = split_values < train_end
    val_mask = (split_values >= train_end) & (split_values < val_end)
    test_mask = split_values >= val_end
    
    splits = {
        'train': df[train_mask].reset_index(drop=True),
        'val': df[val_mask].reset_index(drop=True),
        'test': df[test_mask].reset_index(drop=True),
    }
    
    # Verify no overlap
    train_sources = set(splits['train']['source'])
    val_sources = set(splits['val']['source'])
    test_sources = set(splits['test']['source'])
    
    assert len(train_sources & val_sources) == 0, 'Train/val overlap!'
    assert len(train_sources & test_sources) == 0, 'Train/test overlap!'
    assert len(val_sources & test_sources) == 0, 'Val/test overlap!'
    
    print(f'  Train: {len(splits["train"]):,} | Val: {len(splits["val"]):,} | Test: {len(splits["test"]):,}')
    return splits

print('Splitting function ready.')

## 6. Run Full Pipeline

In [ ]:
cleaner = DataCleaner(CONFIG['cleaning'])
all_reports = {}

for lang_pair in CONFIG['language_pairs']:
    print(f'\n{"="*60}')
    print(f'Processing: {lang_pair}')
    print(f'{"="*60}')
    
    src_lang, tgt_lang = lang_pair.split('-')
    
    # Gather all sources for this pair
    sources = {
        'ai4bharat': ai4bharat_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
        'opus': opus_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
        'custom': custom_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
    }
    if lang_pair == 'hi-en':
        sources['iit_bombay'] = iit_bombay_df
    
    # Step 1: Mix
    ratios = CONFIG['mixing_ratios'][lang_pair]
    mixed_df = mix_datasets(lang_pair, sources, ratios)
    
    if len(mixed_df) == 0:
        print(f'  Skipping {lang_pair} — no data available')
        continue
    
    # Save mixed (before cleaning)
    mixed_path = DATA_MIXED / lang_pair
    mixed_path.mkdir(parents=True, exist_ok=True)
    mixed_df.to_csv(mixed_path / 'mixed.tsv', sep='\t', index=False)
    
    # Step 2: Clean
    print(f'\nCleaning {lang_pair}...')
    cleaned_df = cleaner.clean(mixed_df[['source', 'target']], src_lang, tgt_lang)
    report = cleaner.get_report()
    all_reports[lang_pair] = report
    
    # Step 3: Split
    print(f'\nSplitting {lang_pair}...')
    splits = create_splits(cleaned_df, CONFIG['split_ratios'])
    
    # Save cleaned splits
    clean_path = DATA_CLEANED / lang_pair
    clean_path.mkdir(parents=True, exist_ok=True)
    
    for split_name, split_df in splits.items():
        split_df[['source', 'target']].to_csv(
            clean_path / f'{split_name}.tsv', sep='\t', index=False
        )
    
    # Also create reverse pair (en-XX) by swapping columns
    reverse_pair = f'{tgt_lang}-{src_lang}'
    reverse_path = DATA_CLEANED / reverse_pair
    reverse_path.mkdir(parents=True, exist_ok=True)
    
    for split_name, split_df in splits.items():
        reverse_df = split_df[['target', 'source']].copy()
        reverse_df.columns = ['source', 'target']
        reverse_df.to_csv(reverse_path / f'{split_name}.tsv', sep='\t', index=False)
    
    print(f'\n✅ {lang_pair} + {reverse_pair} — done!')

print(f'\n{"="*60}')
print('All language pairs processed!')

## 7. Cleaning Reports

In [ ]:
# Save cleaning reports
report_path = REPORTS_DIR / 'cleaning_report.json'
with open(report_path, 'w') as f:
    json.dump(all_reports, f, indent=2)

print('📊 Cleaning Report Summary:')
print(f'{"":-<60}')
for lp, report in all_reports.items():
    print(f'\n{lp}:')
    print(f'  Initial:          {report["initial"]:>10,}')
    print(f'  Empty removed:    {report["empty_removed"]:>10,}')
    print(f'  Identical removed:{report["identical_removed"]:>10,}')
    print(f'  Too long removed: {report["too_long_removed"]:>10,}')
    print(f'  Bad ratio removed:{report["bad_ratio_removed"]:>10,}')
    print(f'  Duplicates:       {report["duplicates_removed"]:>10,}')
    print(f'  URL/Email:        {report["url_email_removed"]:>10,}')
    print(f'  Wrong language:   {report["wrong_language_removed"]:>10,}')
    print(f'  Final:            {report["final"]:>10,}')
    print(f'  Retention rate:   {report["retention_rate"]:>10.1%}')

## 8. Verification

In [ ]:
print('🔍 Verifying outputs...\n')
all_ok = True

for lang_pair in CONFIG['language_pairs']:
    src_lang, tgt_lang = lang_pair.split('-')
    
    for direction in [lang_pair, f'{tgt_lang}-{src_lang}']:
        clean_path = DATA_CLEANED / direction
        
        for split in ['train', 'val', 'test']:
            fpath = clean_path / f'{split}.tsv'
            assert fpath.exists(), f'Missing: {fpath}'
            
            df = pd.read_csv(fpath, sep='\t')
            assert len(df) > 0, f'Empty file: {fpath}'
            assert 'source' in df.columns, f'Missing source column: {fpath}'
            assert 'target' in df.columns, f'Missing target column: {fpath}'
            
            # Check no empty values
            empty_src = df['source'].isna().sum() + (df['source'].astype(str).str.strip() == '').sum()
            empty_tgt = df['target'].isna().sum() + (df['target'].astype(str).str.strip() == '').sum()
            assert empty_src == 0, f'Empty sources in {fpath}'
            assert empty_tgt == 0, f'Empty targets in {fpath}'
        
        # Check no overlap between splits
        train_df = pd.read_csv(clean_path / 'train.tsv', sep='\t')
        val_df = pd.read_csv(clean_path / 'val.tsv', sep='\t')
        test_df = pd.read_csv(clean_path / 'test.tsv', sep='\t')
        
        train_src = set(train_df['source'])
        val_src = set(val_df['source'])
        test_src = set(test_df['source'])
        
        assert len(train_src & val_src) == 0, f'Train/val overlap in {direction}'
        assert len(train_src & test_src) == 0, f'Train/test overlap in {direction}'
        
        total = len(train_df) + len(val_df) + len(test_df)
        print(f'  ✅ {direction}: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,} total={total:,}')

print(f'\n✅ All verifications passed!')
print(f'\nOutput directory: {DATA_CLEANED}')
print('These files will be used by Notebook 2 (Training).')